# components

> FastHTML component helpers for the Web Audio API manager

In [ ]:
#| default_exp components

In [ ]:
#| export
import json
from importlib.resources import files as _pkg_files
from typing import Any, List

from fasthtml.common import Input, Script
from starlette.routing import Mount
from starlette.staticfiles import StaticFiles

from cjm_fasthtml_web_audio.models import WebAudioConfig, WebAudioHtmlIds
from cjm_fasthtml_web_audio.js import generate_web_audio_js, DEFAULT_WORKLET_URL

## Audio URLs Input

Hidden input that stores audio file URLs as a JSON array. The JS init function reads from this element to know which files to fetch and decode.

In [ ]:
#| export
def render_audio_urls_input(
    config: WebAudioConfig,    # Instance configuration
    audio_urls: List[str],     # Audio file URLs to load
    oob: bool = False,         # Whether to render as OOB swap
) -> Any:  # Hidden input element with JSON-encoded URLs
    """Render a hidden input storing audio URLs as JSON."""
    return Input(
        type="hidden",
        id=WebAudioHtmlIds.audio_urls_input(config.namespace),
        value=json.dumps(audio_urls),
        hx_swap_oob="true" if oob else None,
    )

## Web Audio Script

Assembles the complete JS into a FastHTML Script element.

In [ ]:
#| export
def render_web_audio_script(
    config: WebAudioConfig,        # Instance configuration
    focus_input_id: str,           # Hidden input ID for focused index
    card_stack_id: str,            # Card stack container ID
    nav_down_btn_id: str = "",     # Nav down button ID (for auto-navigate)
) -> Any:  # Script element with complete Web Audio JS
    """Render the complete Web Audio API script for a configured instance."""
    js = generate_web_audio_js(
        config=config,
        focus_input_id=focus_input_id,
        card_stack_id=card_stack_id,
        nav_down_btn_id=nav_down_btn_id,
    )
    return Script(js)

## Mount Static Assets

Mounts the library's vendored static directory (containing the SoundTouch worklet processor and its license) at a configurable URL path. Host apps call this once at startup when using `enable_speed=True`. Returns the URL of the SoundTouch worklet so consumers can pass it into `WebAudioConfig.worklet_url` if they mount somewhere non-default.

In [ ]:
#| export
# Default mount path; must stay in sync with DEFAULT_WORKLET_URL in cjm_fasthtml_web_audio.js
DEFAULT_STATIC_MOUNT_PATH = "/static/cjm-web-audio"

def mount_web_audio_static(
    app,                                            # FastHTML/Starlette app
    mount_path: str = DEFAULT_STATIC_MOUNT_PATH,    # URL prefix to mount static dir at
) -> str:                                           # URL of the SoundTouch worklet processor
    """Mount the library's vendored static assets (SoundTouch worklet, license).
    
    Inserts the Mount at `app.routes[0]` so it's matched BEFORE any catch-all
    routes FastHTML may have registered. Returns the URL where the SoundTouch
    worklet processor is served; when `mount_path` is the default the URL equals
    `DEFAULT_WORKLET_URL` — so `WebAudioConfig(enable_speed=True)` with no explicit
    `worklet_url` just works.
    """
    static_dir = str(_pkg_files("cjm_fasthtml_web_audio") / "static")
    mount = Mount(
        mount_path,
        app=StaticFiles(directory=static_dir),
        name="cjm-web-audio-static",
    )
    app.routes.insert(0, mount)
    return f"{mount_path.rstrip('/')}/soundtouch-processor.js"

## Initial Speed Sync

Syncs the web-audio JS state (`window._webAudio_{namespace}.playbackSpeed`) to a persisted speed value after render. Works around `generate_state_init` unconditionally resetting `playbackSpeed` to 1.0 every time the script re-executes — the `<option selected>` attribute on a speed selector only restores the dropdown visually, so the actual JS state must be synced explicitly when it should be something other than 1.0.

Typical use: emit alongside a speed selector whenever the current/persisted speed differs from 1.0. Callers on step pages, chrome switches, and HTMX OOB swaps all benefit because the helper polls briefly for `window.set{Ns}Speed` — robust to script execution order between toolbar (caller) and column body (web-audio script).

In [ ]:
#| export
def render_initial_speed_sync(
    config: WebAudioConfig,     # Instance configuration
    speed: float,               # Persisted playback speed (e.g. from step state)
) -> Any:                       # Script element (empty if speed is default or speed is disabled)
    """Render a <Script> that syncs the JS state's playbackSpeed to `speed` after insertion.
    
    Polls briefly (~1 second, 20ms intervals) for `window.set{Ns}Speed` so the helper
    works regardless of whether the web-audio script or the toolbar script runs first.
    Returns an empty Script when `speed == 1.0` (state_init already defaults to 1.0)
    or when `config.enable_speed` is False.
    """
    if not config.enable_speed or speed == 1.0:
        return Script("")
    ns = config.ns
    return Script(f"""
    (function() {{
      var tries = 0;
      var apply = function() {{
        if (window.set{ns}Speed) {{
          window.set{ns}Speed({speed});
        }} else if (tries++ < 50) {{
          setTimeout(apply, 20);
        }}
      }};
      apply();
    }})();
    """)

## Tests

In [ ]:
from fasthtml.common import to_xml, FastHTML

cfg = WebAudioConfig(namespace="demo", indicator_selector=".demo-indicator")

# Test audio URLs input
urls_input = render_audio_urls_input(cfg, ["/audio/file1.mp3", "/audio/file2.mp3"])
html = to_xml(urls_input)
assert 'id="sd-demo-audio-urls"' in html
assert '/audio/file1.mp3' in html
assert '/audio/file2.mp3' in html
assert 'type="hidden"' in html

# Test OOB mode
oob_input = render_audio_urls_input(cfg, ["/audio/file1.mp3"], oob=True)
oob_html = to_xml(oob_input)
assert 'hx-swap-oob="true"' in oob_html

# Test script rendering
script = render_web_audio_script(cfg, focus_input_id="demo-focus", card_stack_id="demo-cs")
script_html = to_xml(script)
assert 'initDemoAudio' in script_html
assert 'playDemoSegment' in script_html

# Default mount path must line up with DEFAULT_WORKLET_URL so enable_speed=True with no
# explicit worklet_url "just works" after mount_web_audio_static(app)
assert DEFAULT_WORKLET_URL == f"{DEFAULT_STATIC_MOUNT_PATH}/soundtouch-processor.js"

# Vendored worklet file is packaged and locatable
static_dir = _pkg_files("cjm_fasthtml_web_audio") / "static"
assert (static_dir / "soundtouch-processor.js").is_file()
assert (static_dir / "LICENSE.soundtouchjs").is_file()

# Mount helper registers a StaticFiles route and returns the correct URL
app = FastHTML()
url = mount_web_audio_static(app)
assert url == DEFAULT_WORKLET_URL
# Custom mount path: returned URL reflects the override
app2 = FastHTML()
url2 = mount_web_audio_static(app2, mount_path="/assets/audio")
assert url2 == "/assets/audio/soundtouch-processor.js"

# Initial speed sync — no-op paths
cfg_no_speed = WebAudioConfig(namespace="plain", indicator_selector=".ind")  # enable_speed=False
no_speed_html = to_xml(render_initial_speed_sync(cfg_no_speed, 2.0))
assert 'setPlainSpeed' not in no_speed_html  # disabled config → no sync script

cfg_speed = WebAudioConfig(namespace="align", indicator_selector=".ind", enable_speed=True)
at_default_html = to_xml(render_initial_speed_sync(cfg_speed, 1.0))
assert 'setAlignSpeed' not in at_default_html  # speed==1.0 → state_init default is correct

# Initial speed sync — active path emits a polling script
sync_html = to_xml(render_initial_speed_sync(cfg_speed, 2.5))
assert 'setAlignSpeed(2.5)' in sync_html
assert 'setTimeout(apply, 20)' in sync_html
assert 'tries++ < 50' in sync_html  # ~1s budget

# Review namespace produces setReviewSpeed
cfg_review = WebAudioConfig(namespace="review", indicator_selector=".ind", enable_speed=True)
review_sync_html = to_xml(render_initial_speed_sync(cfg_review, 1.5))
assert 'setReviewSpeed(1.5)' in review_sync_html

print("All component tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()